In [3]:
%%time

import kplex_dev

timeout_replacement = 1800
excel_path = "kplex_ML_data.xlsx"

cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
cat_df

CPU times: user 1.98 s, sys: 2.32 s, total: 4.31 s
Wall time: 4.92 s


,dataset,n,m,d_avg,d_max,core_max,kcore_list,UMKP,AltRB,kPEX,Pivot_Br,S_Br,k_value
0,MANN-a81,3321.0,5506380.0,1658.0488,3317.0,3280.0,"[(3280,3321)]",1459.5,1800.0,1468.1,1477.4,1504.7,20
1,wikipedia_link_sr,1810.0,1637059.0,904.4525,1809.0,1798.0,"[(1795,1810),(1796,1808),(1797,1804),(1798,1803)]",22.6,18.0,737.7,740.4,738.1,20
2,scc_reality,1332.0,880733.0,661.2110,1331.0,1235.0,"[(1230,1332),(1231,1330),(1232,1309),(1233,126...",1800.0,1214.8,9.4,9.7,9.0,20
3,web-wikipedia_link_en13-all,432.0,93085.0,215.4745,431.0,427.0,"[(427,432)]",239.3,287.1,517.7,504.4,516.7,20
4,livejournal-links,442.0,94435.0,213.6538,441.0,371.0,"[(369,442),(370,409),(371,386)]",1.3,1.6,1.6,1.6,1.6,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,264.0,11792.0,44.6667,150.0,67.0,"[(52,264),(53,262),(54,261),(55,260),(56,256),...",2.0,0.6,0.5,0.5,2.4,2
756,edit-ukwiki,281.0,14183.0,50.4733,280.0,65.0,"[(40,281),(41,279),(42,275),(43,271),(44,268),...",1.2,1.1,0.6,0.6,0.5,2
757,johnson8-4-4,70.0,1855.0,26.5000,53.0,53.0,"[(53,70)]",0.8,0.9,1.1,1.1,0.3,2
758,hamming6-2,64.0,1824.0,28.5000,57.0,57.0,"[(57,64)]",0.2,0.3,0.2,0.2,0.3,2


In [4]:
import kplex_dev

epsilon = 0.001 # need to set !!!

kplex_dev.assign_weights(cat_df, kplex_dev.W_LOG_TIME_RATIO, epsilon)
display(cat_df)
kplex_dev.export_sorted(cat_df)

,dataset,n,m,d_avg,d_max,core_max,kcore_list,UMKP,AltRB,kPEX,Pivot_Br,S_Br,k_value,weight
0,MANN-a81,3321.0,5506380.0,1658.0488,3317.0,3280.0,"[(3280,3321)]",1459.5,1800.0,1468.1,1477.4,1504.7,20,0.209693
1,wikipedia_link_sr,1810.0,1637059.0,904.4525,1809.0,1798.0,"[(1795,1810),(1796,1808),(1797,1804),(1798,1803)]",22.6,18.0,737.7,740.4,738.1,20,3.716819
2,scc_reality,1332.0,880733.0,661.2110,1331.0,1235.0,"[(1230,1332),(1231,1330),(1232,1309),(1233,126...",1800.0,1214.8,9.4,9.7,9.0,20,5.298317
3,web-wikipedia_link_en13-all,432.0,93085.0,215.4745,431.0,427.0,"[(427,432)]",239.3,287.1,517.7,504.4,516.7,20,0.771678
4,livejournal-links,442.0,94435.0,213.6538,441.0,371.0,"[(369,442),(370,409),(371,386)]",1.3,1.6,1.6,1.6,1.6,20,0.207639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,264.0,11792.0,44.6667,150.0,67.0,"[(52,264),(53,262),(54,261),(55,260),(56,256),...",2.0,0.6,0.5,0.5,2.4,2,1.568616
756,edit-ukwiki,281.0,14183.0,50.4733,280.0,65.0,"[(40,281),(41,279),(42,275),(43,271),(44,268),...",1.2,1.1,0.6,0.6,0.5,2,0.875469
757,johnson8-4-4,70.0,1855.0,26.5000,53.0,53.0,"[(53,70)]",0.8,0.9,1.1,1.1,0.3,2,1.299283
758,hamming6-2,64.0,1824.0,28.5000,57.0,57.0,"[(57,64)]",0.2,0.3,0.2,0.2,0.3,2,0.405465


# Expanding kcore-list feature

We will generate these columns for use by conventional models (without a sequential component):
* shell_num: number of k-shells in the k-core list
* core_min: minimum k in the k-core list (note that core_max is already a feature)
* shell_avgSize: average k-shell size
* shell_maxSize: max k-shell size
* shell_minSize: min k-shell size
* shell_stdSize: std of k-shell sizes

In [11]:
import numpy as np

def process_kcore_list(df):
    # Create temporary lists to store the new features
    shell_num = []
    core_min = []
    shell_avgSize = []
    shell_maxSize = []
    shell_minSize = []
    shell_stdSize = []
    
    for lst in df['kcore_list']:
        # String to list
        lst = eval(lst)
        
        # Extract sizes from each tuple in the list
        sizes = [size for (k, size) in lst]
        
        shell_num.append(len(lst))
        core_min.append(lst[0][0])  # First element's k value
        shell_avgSize.append(np.mean(sizes))
        shell_maxSize.append(np.max(sizes))
        shell_minSize.append(np.min(sizes))
        shell_stdSize.append(np.std(sizes))
    
    # Add the new columns to the dataframe
    df['shell_num'] = shell_num
    df['core_min'] = core_min
    df['shell_avgSize'] = shell_avgSize
    df['shell_maxSize'] = shell_maxSize
    df['shell_minSize'] = shell_minSize
    df['shell_stdSize'] = shell_stdSize
    
    # Drop the original column
    df.drop('kcore_list', axis=1, inplace=True)
    
    return df

In [12]:
process_kcore_list(cat_df)

,dataset,n,m,d_avg,d_max,core_max,UMKP,AltRB,kPEX,Pivot_Br,S_Br,k_value,weight,shell_num,core_min,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize
0,MANN-a81,3321.0,5506380.0,1658.0488,3317.0,3280.0,1459.5,1800.0,1468.1,1477.4,1504.7,20,0.209693,1,3280,3321.000000,3321,3321,0.000000
1,wikipedia_link_sr,1810.0,1637059.0,904.4525,1809.0,1798.0,22.6,18.0,737.7,740.4,738.1,20,3.716819,4,1795,1806.250000,1810,1803,2.861381
2,scc_reality,1332.0,880733.0,661.2110,1331.0,1235.0,1800.0,1214.8,9.4,9.7,9.0,20,5.298317,6,1230,1290.333333,1332,1236,35.714921
3,web-wikipedia_link_en13-all,432.0,93085.0,215.4745,431.0,427.0,239.3,287.1,517.7,504.4,516.7,20,0.771678,1,427,432.000000,432,432,0.000000
4,livejournal-links,442.0,94435.0,213.6538,441.0,371.0,1.3,1.6,1.6,1.6,1.6,20,0.207639,3,369,412.333333,442,386,22.983086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,264.0,11792.0,44.6667,150.0,67.0,2.0,0.6,0.5,0.5,2.4,2,1.568616,16,52,215.000000,264,125,56.656862
756,edit-ukwiki,281.0,14183.0,50.4733,280.0,65.0,1.2,1.1,0.6,0.6,0.5,2,0.875469,26,40,235.384615,281,168,34.703222
757,johnson8-4-4,70.0,1855.0,26.5000,53.0,53.0,0.8,0.9,1.1,1.1,0.3,2,1.299283,1,53,70.000000,70,70,0.000000
758,hamming6-2,64.0,1824.0,28.5000,57.0,57.0,0.2,0.3,0.2,0.2,0.3,2,0.405465,1,57,64.000000,64,64,0.000000


In [3]:
# test run

import kplex_dev

timeout_replacement = 1800
excel_path = "kplex_ML_data.xlsx"

cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)

kplex_dev.process_kcore_list_to_statistics(cat_df)
cat_df

,dataset,n,m,d_avg,d_max,core_max,UMKP,AltRB,kPEX,Pivot_Br,S_Br,k_value,shell_num,core_min,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize
0,MANN-a81,3321.0,5506380.0,1658.0488,3317.0,3280.0,1459.5,1800.0,1468.1,1477.4,1504.7,20,1,3280,3321.000000,3321,3321,0.000000
1,wikipedia_link_sr,1810.0,1637059.0,904.4525,1809.0,1798.0,22.6,18.0,737.7,740.4,738.1,20,4,1795,1806.250000,1810,1803,2.861381
2,scc_reality,1332.0,880733.0,661.2110,1331.0,1235.0,1800.0,1214.8,9.4,9.7,9.0,20,6,1230,1290.333333,1332,1236,35.714921
3,web-wikipedia_link_en13-all,432.0,93085.0,215.4745,431.0,427.0,239.3,287.1,517.7,504.4,516.7,20,1,427,432.000000,432,432,0.000000
4,livejournal-links,442.0,94435.0,213.6538,441.0,371.0,1.3,1.6,1.6,1.6,1.6,20,3,369,412.333333,442,386,22.983086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,264.0,11792.0,44.6667,150.0,67.0,2.0,0.6,0.5,0.5,2.4,2,16,52,215.000000,264,125,56.656862
756,edit-ukwiki,281.0,14183.0,50.4733,280.0,65.0,1.2,1.1,0.6,0.6,0.5,2,26,40,235.384615,281,168,34.703222
757,johnson8-4-4,70.0,1855.0,26.5000,53.0,53.0,0.8,0.9,1.1,1.1,0.3,2,1,53,70.000000,70,70,0.000000
758,hamming6-2,64.0,1824.0,28.5000,57.0,57.0,0.2,0.3,0.2,0.2,0.3,2,1,57,64.000000,64,64,0.000000
